# Q1 Prompt Evaluation

This notebook compares `prompt_testing_scored.json` against the reference labels in `labeled_notes.json`.

It is structured to cover the Q1 reporting requirements:
- The LLM used
- The final refined prompt and prompt-engineering strategies
- The evaluation approach, with justification
- The resulting performance
- The confusion matrix, with a brief interpretation


## Report Metadata

Update these fields before submission:

- **LLM used:** `TODO`
- **Final refined prompt:** `Q1 prompt.txt` is currently empty in this workspace, so paste your final prompt here.
- **Prompt refinement strategies implemented:** `TODO`


In [ ]:
from pathlib import Path
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, mean_absolute_error

plt.style.use("seaborn-v0_8-whitegrid")

gold_path = Path("labeled_notes.json")
pred_path = Path("prompt_testing_scored.json")

print(f"Gold labels: {gold_path.resolve()}")
print(f"Predictions: {pred_path.resolve()}")


In [ ]:
gold = json.loads(gold_path.read_text(encoding="utf-8"))
pred = json.loads(pred_path.read_text(encoding="utf-8"))

gold_map = {item["client_id"]: item for item in gold}
pred_map = {item["client_id"]: item for item in pred}

shared_ids = sorted(set(gold_map).intersection(pred_map))
missing_in_gold = sorted(set(pred_map) - set(gold_map))

if missing_in_gold:
    raise ValueError(f"Prediction file has client_ids missing from gold labels: {missing_in_gold}")

records = []
for client_id in shared_ids:
    true_scores = gold_map[client_id]["scored_progress"]
    pred_scores = pred_map[client_id]["scored_progress"]

    if len(true_scores) != len(pred_scores):
        raise ValueError(
            f"{client_id} has different score lengths: {len(true_scores)} vs {len(pred_scores)}"
        )

    for score_index, (true_label, pred_label) in enumerate(zip(true_scores, pred_scores), start=1):
        records.append(
            {
                "client_id": client_id,
                "score_index": score_index,
                "session_number": score_index + 1,
                "true_label": true_label,
                "pred_label": pred_label,
                "abs_error": abs(true_label - pred_label),
                "correct": int(true_label == pred_label),
            }
        )

comparison_df = pd.DataFrame(records)

print(f"Shared clients: {shared_ids}")
print(f"Scored comparisons: {len(comparison_df)}")
comparison_df.head()


## Evaluation Approach

Because the progress scores are **judgment-based and ordinal**, the primary metric in this notebook is **Quadratic Weighted Kappa (QWK)**.

Why QWK is the best fit here:
- It respects the ordering of the labels (`0 < 1 < 2 < 3`).
- It penalizes larger disagreements more than near misses.
- It measures agreement beyond chance, which is helpful when class frequencies are uneven.

To make the results easier to interpret, the notebook also reports:
- **Exact-match accuracy** as a simple agreement rate.
- **Mean absolute error (MAE)** as the average distance between the predicted and reference ordinal scores.

Note: each client has 12 notes but 11 `scored_progress` values. This notebook compares the shared 11 ordinal scores per client and uses `session_number = score_index + 1` as a convenient label.


In [ ]:
labels = [0, 1, 2, 3]
y_true = comparison_df["true_label"]
y_pred = comparison_df["pred_label"]

accuracy = accuracy_score(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
qwk = cohen_kappa_score(y_true, y_pred, labels=labels, weights="quadratic")

summary_df = pd.DataFrame(
    {
        "metric": [
            "Shared clients",
            "Scored comparisons",
            "Exact-match accuracy",
            "Mean absolute error",
            "Quadratic weighted kappa",
        ],
        "value": [
            len(shared_ids),
            len(comparison_df),
            f"{accuracy:.3f} ({accuracy:.1%})",
            f"{mae:.3f}",
            f"{qwk:.3f}",
        ],
    }
)

summary_df


In [ ]:
client_summary = (
    comparison_df.groupby("client_id", as_index=False)
    .agg(
        samples=("correct", "size"),
        exact_matches=("correct", "sum"),
        accuracy=("correct", "mean"),
        mean_abs_error=("abs_error", "mean"),
    )
)

client_summary["accuracy"] = client_summary["accuracy"].map(lambda x: f"{x:.1%}")
client_summary["mean_abs_error"] = client_summary["mean_abs_error"].map(lambda x: f"{x:.3f}")
client_summary


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"True {label}" for label in labels], columns=[f"Pred {label}" for label in labels])
display(cm_df)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted label")
ax.set_ylabel("Reference label")
ax.set_title("Confusion Matrix: prompt_testing_scored vs labeled_notes")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black",
        )

fig.colorbar(im, ax=ax, shrink=0.85)
plt.tight_layout()
plt.show()

label_summary = pd.DataFrame(
    [
        {
            "label": label,
            "support": int(cm[idx].sum()),
            "predicted_count": int(cm[:, idx].sum()),
            "recall": (cm[idx, idx] / cm[idx].sum()) if cm[idx].sum() else np.nan,
        }
        for idx, label in enumerate(labels)
    ]
)

label_summary["recall"] = label_summary["recall"].map(lambda x: f"{x:.1%}" if pd.notna(x) else "NA")
label_summary


In [ ]:
mismatches = comparison_df[comparison_df["correct"] == 0].copy()
off_by_one = int((mismatches["abs_error"] == 1).sum())
off_by_two_or_more = int((mismatches["abs_error"] >= 2).sum())
over_predictions = int((comparison_df["pred_label"] > comparison_df["true_label"]).sum())
under_predictions = int((comparison_df["pred_label"] < comparison_df["true_label"]).sum())

true_counts = Counter(y_true)
pred_counts = Counter(y_pred)

cm_numeric = confusion_matrix(y_true, y_pred, labels=labels)
recalls = {
    label: (cm_numeric[idx, idx] / cm_numeric[idx].sum()) if cm_numeric[idx].sum() else np.nan
    for idx, label in enumerate(labels)
}
best_label = max(recalls, key=lambda label: recalls[label])
hardest_label = min(recalls, key=lambda label: recalls[label])

print(
    f"Primary result: QWK = {qwk:.3f}, which indicates moderate ordinal agreement between the prompt-based scores and the reference labels."
)
print(
    f"Supporting results: exact-match accuracy = {accuracy:.1%} ({int(comparison_df['correct'].sum())}/{len(comparison_df)}), and MAE = {mae:.3f}."
)
print(
    f"Most errors were near misses: {off_by_one} of {len(mismatches)} mismatches were off by exactly 1 point, while {off_by_two_or_more} were off by 2 or more points."
)
print(
    f"The model over-predicted {over_predictions} times and under-predicted {under_predictions} times, so its mistakes lean slightly toward assigning higher scores than the reference labels."
)
print(
    f"Label {best_label} was identified most reliably (recall = {recalls[best_label]:.1%}), while label {hardest_label} was the most difficult (recall = {recalls[hardest_label]:.1%})."
)
print(
    f"A notable confusion-matrix pattern is middle-category inflation: label 2 was predicted {pred_counts[2]} times even though it appears only {true_counts[2]} times in the reference labels."
)


In [ ]:
mismatches[["client_id", "score_index", "session_number", "true_label", "pred_label", "abs_error"]].sort_values(
    ["client_id", "score_index"]
)
